# Chapter 25 — Anomaly Detection with AI
## From "CPU > 90%" to "This Device Is Behaving Unusually for a Tuesday Morning"

Traditional monitoring is **threshold-based**:
- CPU over 90%: alert.
- Interface utilization over 80%: alert.

The problem: the BGP route reflector that idles at **60% CPU is perfectly healthy**.
The access switch that normally runs at 5% CPU hitting **40% is a real problem** —
but your threshold fires at 90%, so nothing triggers.

**Anomaly detection inverts this:**
Instead of "is this above my threshold?"
→ "is this behaving differently from how it *normally* behaves?"

The system learns what normal looks like — **per device, per time of day, per day of week** —
and alerts when behavior deviates. No manual threshold configuration.

| Technique | What It Detects | When to Use |
|-----------|----------------|-------------|
| **Z-Score** | Single-metric spikes | Normally-distributed metrics |
| **MAD** | Single-metric spikes (robust) | Heavy-tailed network metrics ✅ |
| **STL Decomposition** | Deviations from seasonal baseline | Metrics with daily/weekly patterns |
| **Isolation Forest** | Multivariate anomalies | Many metrics together |
| **LLM Explainer** | Context-aware analysis | Root cause explanation |

> **No GPU, no sklearn, no tensorflow.** All demos use pure Python math + Anthropic API.

### Install
```
pip install anthropic
```


## Setup — Math Helpers and Simulated Network Metrics

In [ ]:
import json, re, math, random, time
from collections import defaultdict
from anthropic import Anthropic

# ── API Key Setup ─────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    api_key = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=api_key)

# ── Model IDs ─────────────────────────────────────────────────────────────────
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=HAIKU, system="You are a network operations AI assistant."):
    r = client.messages.create(
        model=model, max_tokens=1024, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# ── Pure-Python statistics helpers ────────────────────────────────────────────
def mean(data):
    return sum(data) / len(data) if data else 0.0

def std(data):
    if len(data) < 2: return 0.0
    m = mean(data)
    return math.sqrt(sum((x - m)**2 for x in data) / (len(data) - 1))

def median(data):
    s = sorted(data)
    n = len(s)
    return (s[n//2 - 1] + s[n//2]) / 2 if n % 2 == 0 else s[n//2]

def mad(data):
    """Median Absolute Deviation - robust measure of dispersion."""
    m = median(data)
    return median([abs(x - m) for x in data])

def moving_average(data, window):
    """Simple moving average with given window size."""
    result = []
    for i in range(len(data)):
        start = max(0, i - window + 1)
        result.append(mean(data[start:i+1]))
    return result

# ── Simulate 7 days of CPU telemetry (hourly) - 168 data points ──────────────
random.seed(42)

def simulate_cpu_timeseries(days=7):
    """
    Realistic CPU time series with:
    - Trend: slow upward drift (more devices added each day)
    - Seasonality: high during business hours (9-17), low at night, lower weekends
    - Noise: random normal variation
    - Anomalies: 3 injected incidents
    """
    series = []
    labels = []   # True anomaly labels (for evaluation)

    # Injected anomalies: (hour_index, magnitude)
    anomaly_hours = {38: 45, 75: 60, 140: 55}   # hours into the week

    for day in range(days):
        is_weekend = day >= 5
        for hour in range(24):
            idx = day * 24 + hour
            # Trend: +0.05% per hour baseline drift
            trend = 12.0 + idx * 0.05
            # Seasonality: business hours peak
            if not is_weekend and 9 <= hour <= 17:
                seasonal = 15.0 + 5.0 * math.sin(math.pi * (hour - 9) / 8)
            elif is_weekend:
                seasonal = -5.0   # lower on weekends
            else:
                seasonal = -8.0   # low overnight
            # Noise
            noise = random.gauss(0, 2.5)
            value = max(0, trend + seasonal + noise)
            # Inject anomaly
            if idx in anomaly_hours:
                value += anomaly_hours[idx]
                labels.append(True)
            else:
                labels.append(False)
            series.append(round(value, 1))

    return series, labels


cpu_series, true_anomalies = simulate_cpu_timeseries()

# Label hours for display
hours = [f"D{i//24+1}H{i%24:02d}" for i in range(len(cpu_series))]

print(f"Simulated CPU time series: {len(cpu_series)} hourly data points (7 days)")
print(f"True anomalies injected: {sum(true_anomalies)} (hours {[i for i,a in enumerate(true_anomalies) if a]})")
print(f"CPU range: {min(cpu_series):.1f}% - {max(cpu_series):.1f}%")
print(f"\nFirst 24 hours (Day 1):")
for i, (h, v) in enumerate(zip(hours[:24], cpu_series[:24])):
    bar = chr(9608) * int(v / 3)
    flag = " <- ANOMALY" if true_anomalies[i] else ""
    print(f"  {h}: {v:5.1f}% {bar}{flag}")


---
## Demo 1 — Time Series Decomposition: Operate on the Residual

**Problem:** If you run anomaly detection on raw CPU values, you will fire an alert
every Monday morning when CPU rises from the weekend low to the business-day high.
That is not an anomaly — that is Monday.

**Solution:** Decompose the time series into three components:
```
Raw metric  =  Trend  +  Seasonality  +  Residual

Trend:       slow upward drift (capacity growth)
Seasonality: daily/weekly business pattern
Residual:    the noise — anomalies live here
```

**Anomaly detection must operate on the residual**, not the raw metric.
The residual is what remains after removing *expected* behavior.

This is why `STL decomposition` (Seasonal-Trend using Loess) is the standard.
Here we implement the core idea using simple moving averages.


In [ ]:
# ── Time Series Decomposition ─────────────────────────────────────────────────

def decompose_timeseries(series, period=24, trend_window=24):
    """
    Decompose a time series into Trend + Seasonal + Residual.

    Steps:
    1. Trend: moving average over trend_window
    2. Detrended: series - trend
    3. Seasonal: average detrended value per period position (hour-of-day)
    4. Residual: detrended - seasonal
    """
    n = len(series)

    # Step 1: Trend via moving average
    trend = moving_average(series, trend_window)

    # Step 2: Detrended series
    detrended = [series[i] - trend[i] for i in range(n)]

    # Step 3: Seasonal pattern — average value per hour-of-day slot
    seasonal_avg = defaultdict(list)
    for i, v in enumerate(detrended):
        slot = i % period
        seasonal_avg[slot].append(v)
    seasonal_mean = {slot: mean(vals) for slot, vals in seasonal_avg.items()}

    seasonal = [seasonal_mean[i % period] for i in range(n)]

    # Step 4: Residual = what's left after removing trend + seasonal
    residual = [detrended[i] - seasonal[i] for i in range(n)]

    return trend, seasonal, residual


trend, seasonal, residual = decompose_timeseries(cpu_series, period=24, trend_window=24)

print("TIME SERIES DECOMPOSITION")
print("=" * 60)
print(f"\nComponent ranges:")
print(f"  Raw series : {min(cpu_series):.1f}% – {max(cpu_series):.1f}%")
print(f"  Trend      : {min(trend):.1f}% – {max(trend):.1f}%")
print(f"  Seasonal   : {min(seasonal):.1f}% – {max(seasonal):.1f}%")
print(f"  Residual   : {min(residual):.1f}% – {max(residual):.1f}%")

print(f"\nKey insight: The residual range is MUCH smaller than the raw range.")
print(f"  An anomaly in the residual is a TRUE deviation from the expected pattern.")
print(f"  An anomaly in the raw series might just be 'Monday morning.'")

# Show decomposition around injected anomaly (hour 38 = Day 2, Hour 14)
print(f"\nDecomposition at hour 38 (injected +45% anomaly):")
print(f"  Raw value  : {cpu_series[38]:.1f}%")
print(f"  Trend      : {trend[38]:.1f}%")
print(f"  Seasonal   : {seasonal[38]:.1f}%")
print(f"  Residual   : {residual[38]:.1f}%   ← anomaly score lives here")

print(f"\nDecomposition at hour 39 (normal business hour):")
print(f"  Raw value  : {cpu_series[39]:.1f}%")
print(f"  Trend      : {trend[39]:.1f}%")
print(f"  Seasonal   : {seasonal[39]:.1f}%")
print(f"  Residual   : {residual[39]:.1f}%   ← small, normal")


---
## Demo 2 — Z-Score vs MAD: Why Robust Statistics Matter

**Z-Score**: `z = (x - mean) / std`

The problem for network data: a single spike **inflates both the mean and std**.
After one incident, the detector calibrates itself to *expect* incidents.
Your baseline drifts — future spikes look less anomalous.

**MAD (Median Absolute Deviation)**: `score = 0.6745 * (x - median) / MAD`

The median barely moves when one value spikes.
MAD stays anchored to typical behavior.

```
Normal WAN latency: 20ms
Backup spike:       500ms once/week

Z-score baseline drifts: mean rises to 35ms, std inflates
  → next 500ms spike scores lower → eventually missed

MAD baseline stays: median = 20ms
  → every 500ms spike scores equally high → never missed
```

Network metrics are **heavy-tailed** (lots of near-zero, occasional large spikes).
MAD is the right tool.


In [ ]:
# ── Z-Score vs MAD Comparison ─────────────────────────────────────────────────

def zscore_detect(data, threshold=3.0):
    """Standard Z-score anomaly detection."""
    m = mean(data)
    s = std(data)
    if s == 0:
        return [0.0] * len(data), []
    scores = [(x - m) / s for x in data]
    anomalies = [i for i, z in enumerate(scores) if abs(z) > threshold]
    return scores, anomalies

def mad_detect(data, threshold=3.5):
    """
    Modified Z-score using MAD — robust to outliers.
    Constant 0.6745 makes MAD equivalent to std for normally distributed data.
    """
    m = median(data)
    mad_val = mad(data)
    if mad_val == 0:
        return [0.0] * len(data), []
    scores = [0.6745 * (x - m) / mad_val for x in data]
    anomalies = [i for i, z in enumerate(scores) if abs(z) > threshold]
    return scores, anomalies


# Demonstrate on the residual (correct approach) and raw series (wrong approach)
print("COMPARISON: Z-Score vs MAD")
print("=" * 60)

# ── On raw series (naive, wrong approach) ─────────────────────────────────────
print("\n[Wrong approach] Z-Score on raw CPU values:")
raw_zscores, raw_z_anomalies = zscore_detect(cpu_series, threshold=3.0)
true_indices = [i for i, a in enumerate(true_anomalies) if a]
print(f"  True anomaly hours : {true_indices}")
print(f"  Z-score detections : {raw_z_anomalies}")
raw_precision = len(set(raw_z_anomalies) & set(true_indices)) / max(len(raw_z_anomalies), 1)
raw_recall    = len(set(raw_z_anomalies) & set(true_indices)) / max(len(true_indices), 1)
print(f"  Precision: {raw_precision:.0%}  Recall: {raw_recall:.0%}")
print(f"  (Many false positives — Monday mornings trigger the threshold)")

# ── On residual (correct approach) ────────────────────────────────────────────
print("\n[Correct approach] Z-Score on residual:")
res_zscores, res_z_anomalies = zscore_detect(residual, threshold=3.0)
print(f"  True anomaly hours : {true_indices}")
print(f"  Z-score detections : {res_z_anomalies}")
res_z_precision = len(set(res_z_anomalies) & set(true_indices)) / max(len(res_z_anomalies), 1)
res_z_recall    = len(set(res_z_anomalies) & set(true_indices)) / max(len(true_indices), 1)
print(f"  Precision: {res_z_precision:.0%}  Recall: {res_z_recall:.0%}")

print("\n[Best approach] MAD on residual (robust to heavy-tailed network data):")
mad_scores, mad_anomalies = mad_detect(residual, threshold=3.5)
print(f"  True anomaly hours : {true_indices}")
print(f"  MAD detections     : {mad_anomalies}")
mad_precision = len(set(mad_anomalies) & set(true_indices)) / max(len(mad_anomalies), 1)
mad_recall    = len(set(mad_anomalies) & set(true_indices)) / max(len(true_indices), 1)
print(f"  Precision: {mad_precision:.0%}  Recall: {mad_recall:.0%}")

print("\n" + "─" * 60)
print("Why MAD wins for network data:")
print(f"  Residual median : {median(residual):.2f}%   (stable — not influenced by spikes)")
print(f"  Residual mean   : {mean(residual):.2f}%   (drifts after incidents)")
print(f"  MAD value       : {mad(residual):.2f}%   (robust baseline spread)")
print(f"  Std deviation   : {std(residual):.2f}%   (inflated by spike history)")


---
## Demo 3 — Isolation Forest from Scratch: Catch Multivariate Anomalies

Individual thresholds miss this scenario:
- CPU: 45% (below 90% threshold — no alert)
- Memory: 62% (below 80% threshold — no alert)
- BGP peers: dropped from 12 to 8 (below minimum threshold — no alert)
- Interface errors: 0.02% (below error threshold — no alert)

**No individual metric triggers. But together they scream "something is wrong."**

**Isolation Forest** catches this. The insight:
*Anomalous data points are rare and different — they are isolated in very few random splits.*

```
Normal data point: needs many splits to isolate (it's surrounded by similar points)
Anomalous point:   isolated in 2-3 splits (it's alone in feature space)

Anomaly score = 1 / (average path length to isolation)
Short path → easy to isolate → high anomaly score
```


In [ ]:
# ── Isolation Forest from Scratch ────────────────────────────────────────────

class IsolationTree:
    """A single random isolation tree."""

    def __init__(self, max_depth=8):
        self.max_depth = max_depth
        self.split_feature = None
        self.split_value   = None
        self.left  = None
        self.right = None
        self.size  = 0   # number of data points at this node

    def fit(self, data: list[list[float]], depth=0):
        """Build the tree by random splits."""
        self.size = len(data)
        n_features = len(data[0]) if data else 0

        # Stop when we can't split anymore
        if len(data) <= 1 or depth >= self.max_depth or n_features == 0:
            return self

        # Pick a random feature and random split within its range
        self.split_feature = random.randint(0, n_features - 1)
        col = [row[self.split_feature] for row in data]
        col_min, col_max = min(col), max(col)
        if col_min == col_max:
            return self
        self.split_value = random.uniform(col_min, col_max)

        left_data  = [row for row in data if row[self.split_feature] < self.split_value]
        right_data = [row for row in data if row[self.split_feature] >= self.split_value]

        self.left  = IsolationTree(self.max_depth).fit(left_data,  depth + 1)
        self.right = IsolationTree(self.max_depth).fit(right_data, depth + 1)
        return self

    def path_length(self, point: list[float], depth=0) -> float:
        """How many splits does it take to isolate this point?"""
        # Reached a leaf — estimate remaining path
        if self.split_feature is None or self.size <= 1:
            return depth + self._c(self.size)
        if point[self.split_feature] < self.split_value:
            return self.left.path_length(point, depth + 1)
        else:
            return self.right.path_length(point, depth + 1)

    def _c(self, n) -> float:
        """Expected path length for a subtree with n points."""
        if n <= 1: return 0.0
        return 2 * (math.log(n - 1) + 0.5772) - 2 * (n - 1) / n


class IsolationForest:
    """Ensemble of isolation trees for robust anomaly detection."""

    def __init__(self, n_trees=50, sample_size=64, max_depth=8):
        self.n_trees     = n_trees
        self.sample_size = sample_size
        self.max_depth   = max_depth
        self.trees       = []
        self._c_n        = 1.0

    def fit(self, data: list[list[float]]):
        n = len(data)
        self._c_n = IsolationTree()._c(min(self.sample_size, n))
        for _ in range(self.n_trees):
            sample = random.sample(data, min(self.sample_size, n))
            tree   = IsolationTree(self.max_depth).fit(sample)
            self.trees.append(tree)
        return self

    def anomaly_score(self, point: list[float]) -> float:
        """
        Score between 0 and 1.
        Score > 0.6 → anomalous
        Score < 0.5 → normal
        """
        avg_path = mean([t.path_length(point) for t in self.trees])
        return 2 ** (-avg_path / self._c_n)

    def predict(self, data: list[list[float]], threshold=0.60) -> list[bool]:
        return [self.anomaly_score(p) > threshold for p in data]


# ── Simulate multivariate device telemetry ────────────────────────────────────
random.seed(7)

# Normal device: [cpu%, mem%, bgp_peers, error_rate_pct]
def normal_device():
    return [
        random.gauss(20, 5),    # CPU: ~20%
        random.gauss(45, 8),    # Memory: ~45%
        random.gauss(12, 1),    # BGP peers: ~12
        random.gauss(0.01, 0.005),  # Error rate: ~0.01%
    ]

# Anomalous device: each metric alone looks OK, combination is wrong
def anomalous_device():
    return [
        random.gauss(45, 3),    # CPU elevated but below threshold
        random.gauss(62, 3),    # Memory elevated but below threshold
        random.gauss(8, 1),     # BGP peers dropped (missing 4)
        random.gauss(0.02, 0.005),  # Errors slightly elevated
    ]

# Build dataset: 90 normal + 10 anomalous
normal_data = [normal_device() for _ in range(90)]
anomaly_data = [anomalous_device() for _ in range(10)]
all_data = normal_data + anomaly_data
true_labels = [False] * 90 + [True] * 10

# Train and score
print("ISOLATION FOREST — Multivariate Anomaly Detection")
print("=" * 60)
print("Feature vector: [cpu%, memory%, bgp_peers, interface_error%]")
print()

forest = IsolationForest(n_trees=50, sample_size=64, max_depth=8)
forest.fit(normal_data)   # Train ONLY on normal data

scores = [forest.anomaly_score(p) for p in all_data]
predictions = [s > 0.60 for s in scores]

detected = sum(1 for p, t in zip(predictions, true_labels) if p and t)
fp = sum(1 for p, t in zip(predictions, true_labels) if p and not t)

print(f"Results:")
print(f"  True anomalies    : 10")
print(f"  Detected          : {detected}/10")
print(f"  False positives   : {fp}")
print(f"  Precision         : {detected/(detected+fp)*100:.0f}%")

print(f"\nSample anomaly scores:")
print(f"  Normal devices (avg)  : {mean(scores[:10]):.3f}")
print(f"  Anomaly devices (avg) : {mean(scores[90:]):.3f}")
print(f"  Threshold             : 0.60")
print()
print("Key insight: No individual threshold would have caught these.")
print("CPU=45%, Mem=62%, BGP=8 peers, Errors=0.02% — each within 'acceptable' range.")
print("Together they form an anomalous combination the forest isolates quickly.")


---
## Demo 4 — LLM Anomaly Explainer: From Score to Actionable Alert

A score of 0.73 means nothing to an on-call engineer at 2 AM.

The LLM converts anomaly scores + device context into:
- **Plain-English explanation**: what is unusual and why it matters
- **Probable causes**: ranked from most to least likely
- **Recommended actions**: specific, ordered steps

This is the bridge between "the math detected something" and
"here is what to do about it."

The LLM also handles **contextual severity** — the same anomaly means
different things depending on:
- Is this device in a maintenance window?
- Is this the only path to a site?
- Did this same device anomaly-detect 3 times this week?


In [ ]:
# ── LLM Anomaly Explainer ─────────────────────────────────────────────────────

# Simulated device context (from CMDB + topology)
DEVICE_CONTEXT = {
    "CORE-R1": {
        "role":     "Core Router — only path to BRANCH-SITE-03",
        "site":     "Budapest HQ",
        "vendor":   "Cisco IOS-XE 17.9",
        "bgp_peers_expected": 12,
        "maintenance_window": None,
        "recent_changes":     "None in last 7 days",
        "anomaly_history":    "2 anomalies in past 30 days",
    }
}

def explain_anomaly(device: str, metrics: dict,
                    anomaly_score: float, context: dict) -> str:
    """
    Use Sonnet to explain what the anomaly means and what to do.
    """
    prompt = f"""A network device has been flagged with an anomaly score of {anomaly_score:.2f} (threshold: 0.60).

Device: {device}
Context: {json.dumps(context, indent=2)}

Current metrics vs. normal baseline:
{json.dumps(metrics, indent=2)}

Explain this anomaly to an on-call network engineer:

ANOMALY SUMMARY:
  (what is unusual — compare each metric to normal)

SEVERITY ASSESSMENT:
  (P1/P2/P3, and why given the device's role)

PROBABLE CAUSES (ranked):
  1. (most likely)
  2.
  3.

RECOMMENDED ACTIONS:
  Immediate (next 5 min):
    1. ...
  Short-term (next 30 min):
    1. ...

RISK IF IGNORED:
  (what happens if we wait)
"""
    return ask(prompt, model=SONNET,
               system="You are a senior NOC engineer. Write clear, specific, actionable analysis.")


# Build a realistic anomaly scenario
device_name = "CORE-R1"
current_metrics = {
    "cpu_percent":      {"current": 47.2, "normal_avg": 19.8, "deviation": "+138%"},
    "memory_percent":   {"current": 63.1, "normal_avg": 44.5, "deviation": "+42%"},
    "bgp_peers_active": {"current": 8,    "normal_avg": 12,   "deviation": "-33%"},
    "interface_errors_pct": {"current": 0.021, "normal_avg": 0.010, "deviation": "+110%"},
    "ospf_spf_runs_per_hour": {"current": 47, "normal_avg": 2, "deviation": "+2250%"},
}
anomaly_score = 0.81

print(f"Device: {device_name}")
print(f"Anomaly score: {anomaly_score:.2f}  (threshold: 0.60)")
print()
print("Metrics:")
for k, v in current_metrics.items():
    print(f"  {k}: {v['current']} (normal: {v['normal_avg']}, change: {v['deviation']})")

print()
print("=" * 60)
print("LLM ANOMALY EXPLANATION")
print("=" * 60)
explanation = explain_anomaly(
    device_name,
    current_metrics,
    anomaly_score,
    DEVICE_CONTEXT[device_name]
)
print(explanation)


---
## Demo 5 — Full Anomaly Detection Pipeline

The complete production pipeline — from raw metric stream to alert:

```
Metric telemetry stream (every 60 seconds)
        │
   [Decompose]     → Remove trend + seasonality → get residual
        │
   [MAD Score]     → Robust anomaly score per metric
        │
   [Isolation Forest] → Multivariate check (catches what MAD misses)
        │
   [Threshold Gate]   → MAD > 3.5 OR forest > 0.60 → flag
        │
   [LLM Explainer]    → Context-aware alert with ranked causes + actions
        │
   NOC Alert (P1/P2/P3)
```

Cost model:
- MAD + Isolation Forest: ~$0 (pure math)
- LLM explanation: only runs when an anomaly is detected (~1-5 times/day)
- Daily LLM cost for 500-device network: < $0.05


In [ ]:
# ── Full Anomaly Detection Pipeline ──────────────────────────────────────────

def pipeline_score_metric(series: list[float],
                           period: int = 24,
                           trend_window: int = 24,
                           mad_threshold: float = 3.5) -> list[dict]:
    """
    Full single-metric anomaly scoring:
    1. Decompose into residual
    2. MAD-score each residual point
    Returns: list of {hour, raw, residual, mad_score, flagged}
    """
    _, _, residual = decompose_timeseries(series, period, trend_window)
    mad_scores, _ = mad_detect(residual, threshold=mad_threshold)

    results = []
    for i, (raw, res, score) in enumerate(zip(series, residual, mad_scores)):
        results.append({
            "hour": i,
            "raw": raw,
            "residual": round(res, 2),
            "mad_score": round(abs(score), 2),
            "flagged": abs(score) > mad_threshold,
        })
    return results


def full_pipeline(device: str,
                  metric_streams: dict[str, list[float]],
                  device_ctx: dict,
                  forest: IsolationForest,
                  mad_threshold: float = 3.5,
                  forest_threshold: float = 0.60) -> list[dict]:
    """
    Multi-metric pipeline:
    1. MAD scoring per metric
    2. Build feature vector for latest point
    3. Isolation Forest score
    4. Combine: flag if either method detects anomaly
    5. LLM explain flagged hours
    """
    print(f"Running anomaly pipeline for: {device}")
    print(f"Metrics: {list(metric_streams.keys())}")
    print(f"Time window: {len(next(iter(metric_streams.values())))} hours")
    print("=" * 60)

    # Step 1: MAD score each metric
    per_metric_scores = {}
    for metric, series in metric_streams.items():
        per_metric_scores[metric] = pipeline_score_metric(series, mad_threshold=mad_threshold)

    # Step 2 & 3: Isolation Forest on latest 24h
    n = len(next(iter(metric_streams.values())))
    alerts = []

    for hour in range(24, n):   # skip first 24h (warmup)
        # Build feature vector: current raw value for each metric
        feature_vec = [metric_streams[m][hour] for m in metric_streams]
        if_score = forest.anomaly_score(feature_vec)

        # Gather MAD scores for this hour
        mad_flags = {m: per_metric_scores[m][hour]["flagged"]
                     for m in metric_streams}
        any_mad_flag = any(mad_flags.values())

        if any_mad_flag or if_score > forest_threshold:
            alert = {
                "hour": hour,
                "hour_label": f"D{hour//24+1}H{hour%24:02d}",
                "if_score": round(if_score, 3),
                "mad_flags": mad_flags,
                "mad_scores": {m: per_metric_scores[m][hour]["mad_score"]
                               for m in metric_streams},
                "raw_values": {m: metric_streams[m][hour] for m in metric_streams},
                "detection_method": "BOTH" if any_mad_flag and if_score > forest_threshold
                                    else ("MAD" if any_mad_flag else "ISOLATION_FOREST"),
            }
            alerts.append(alert)

    print(f"\nDetected {len(alerts)} anomaly event(s) in {n-24} scored hours:")
    for a in alerts:
        print(f"\n  [{a['hour_label']}] Score={a['if_score']:.3f} Method={a['detection_method']}")
        for m, v in a["raw_values"].items():
            flag = "⚠" if a["mad_flags"].get(m) else " "
            print(f"    {flag} {m}: {v:.1f}  (MAD score: {a['mad_scores'][m]:.1f})")

    # Step 5: LLM explain the worst anomaly
    if alerts:
        worst = max(alerts, key=lambda a: a["if_score"])
        print(f"\n[LLM Explanation for worst anomaly at {worst['hour_label']}]")
        metrics_for_llm = {
            m: {"current": worst["raw_values"][m],
                "mad_score": worst["mad_scores"][m],
                "flagged": worst["mad_flags"][m]}
            for m in metric_streams
        }
        explanation = explain_anomaly(device, metrics_for_llm, worst["if_score"], device_ctx)
        print(explanation)

    return alerts


# Run the full pipeline
metric_streams = {
    "cpu_pct":      cpu_series,
    "mem_pct":      [random.gauss(45, 6) + (20 if true_anomalies[i] else 0)
                     for i in range(len(cpu_series))],
    "bgp_peers":    [12 + random.gauss(0, 0.5) + (-4 if true_anomalies[i] else 0)
                     for i in range(len(cpu_series))],
    "error_pct":    [0.01 + abs(random.gauss(0, 0.003)) + (0.02 if true_anomalies[i] else 0)
                     for i in range(len(cpu_series))],
}

# Re-train forest on normal data (first 5 days, no anomalies)
normal_vectors = [
    [metric_streams[m][i] for m in metric_streams]
    for i in range(5*24)
]
pipeline_forest = IsolationForest(n_trees=40, sample_size=64).fit(normal_vectors)

alerts = full_pipeline(
    device="CORE-R1",
    metric_streams=metric_streams,
    device_ctx=DEVICE_CONTEXT["CORE-R1"],
    forest=pipeline_forest,
)


---
## Summary — From Thresholds to Contextual Anomaly Detection

### Why Thresholds Fail

```
Threshold: CPU > 90%
Reality:   Route reflector idles at 60% — no alert needed
           Access switch at 40% is a real problem — threshold doesn't fire

Anomaly detection: learns per-device, per-hour baseline
                   alerts when behavior changes, not just when it crosses a number
```

### The Three-Layer Detection Stack

| Layer | Method | Catches |
|-------|--------|---------|
| 1 — Decompose | STL / Moving Average | Remove trend + seasonality before scoring |
| 2 — Statistical | MAD (robust z-score) | Single-metric spikes in residual |
| 3 — Multivariate | Isolation Forest | Combined metric anomalies under individual thresholds |

### Why MAD, Not Z-Score

| | Z-Score | MAD |
|-|---------|-----|
| Sensitive to outliers? | Yes — spikes inflate baseline | No — median is robust |
| Works for heavy-tailed data? | No | Yes ✅ |
| Network suitability | Poor | Excellent ✅ |

### Why Isolation Forest

```
CPU=45% → below 90% threshold → no alert
Memory=62% → below 80% threshold → no alert
BGP peers dropped 33% → no threshold configured → no alert

Isolation Forest: this combination is easy to isolate → ANOMALY SCORE: 0.81
```

### Cost Model (500-device network)

| Step | Volume | Cost |
|------|--------|------|
| Decompose + MAD | All metrics, every minute | ~$0 |
| Isolation Forest | All metrics, every minute | ~$0 |
| LLM explanation | 1-5 anomalies/day | ~$0.05/day |

### What's Next

- **Chapter 26**: Predictive Capacity Planning — using anomaly trends and
  time-series forecasting to predict *when* a device will hit capacity,
  before it becomes an incident

> The shift from threshold monitoring to anomaly detection is the shift
> from "waiting for the alarm to go off" to "knowing something is wrong
> before users notice." That's the difference between reactive and proactive NOC.
